# Part 0 — Foundations: math and ML from zero

This notebook builds the minimum math intuition you need before RL, robotics, and foundation models: vectors, matrices, probability, gradients, and optimization. The goal is not to become a mathematician first; the goal is to recognize the tools when they appear in policies, filters, controllers, and neural networks.

**Learning style:** mechanisms first → frameworks second → real systems third. The notebook is intentionally slow, explicit, and beginner-friendly.

In [ ]:
# Setup: run this first.
# Works from the repository root. In Colab, clone the repo first, then run from inside it.
from pathlib import Path
import sys, math, random
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    print('Tip: run this notebook from the repository root, or clone the repo in Colab first.')
sys.path.insert(0, str(ROOT))
print('Working directory:', ROOT)

## Notebook type and ordered study structure

**Notebook type:** Type A — core mechanism notebook. This should be studied slowly because the mechanism appears again and again across the whole repo.

**Your objective in this notebook:** Build the math reflexes needed for every later topic: vectors, matrices, probability, gradients, and optimization.

Use this exact order every time:

1. **Why this matters** — identify the real problem this topic solves.
2. **Mental model** — explain the idea in plain language before symbols.
3. **Math mechanism** — write the smallest formula and define every symbol.
4. **From-scratch code** — run/read the simple implementation slowly.
5. **Line-by-line explanation** — trace inputs, internal variables, update rule, and outputs.
6. **Debug/visualize** — print shapes, values, curves, maps, or weights so behavior is visible.
7. **Framework version** — map the mechanism to a real library/API.
8. **Scratch → framework mapping** — write what the framework hides and what it exposes.
9. **Real-system role** — place the topic inside robotics, driving, drones, manipulation, or VLA.
10. **Failure modes** — list how it breaks and how you would notice.
11. **Exercises** — change parameters, break the example, and explain the result.
12. **Mini-project** — build a small artifact you can keep.
13. **Next step** — choose the next notebook or tool.

**Mental model for this topic:** Learning is repeated error correction: predict, measure error, compute a direction of improvement, update.

**Core math / mechanism to keep in mind:** Gradient descent: theta <- theta - alpha * grad L(theta).

**Recommended debug habit:** after every code cell, ask “what are the inputs, what changed, and what would be unsafe or wrong in a real robot?”

## From-scratch focus and code-reading checklist

**Scratch focus:** Manual line fitting with gradients, then the same loop in PyTorch.

When reading code in this notebook or the matching repo module, trace it like this:

| Step | Question to answer |
|---|---|
| Input | What is the state, observation, tensor, point, reward, or measurement? |
| Representation | Is it a scalar, vector, matrix, image, point cloud, token sequence, or action? |
| Mechanism | Which line implements the math/update rule? |
| Parameters | Which numbers are hyperparameters, physical constants, or learned weights? |
| Output | What changed after the step? |
| Debug signal | What should I print/plot to know it is working? |

Do not move to the framework/API version until you can explain the scratch version without reading the code comments.

## Visualization and debugging ideas

Use at least one of these while studying:

- Print tensor/vector shapes before and after the core operation.
- Print the first few values before and after an update.
- Plot a curve when there is learning, control, filtering, planning, or optimization.
- Draw frames, maps, paths, sensor rays, or attention matrices when geometry is involved.
- Change one parameter at a time and predict the effect before running.

For this notebook, a useful first visualization/debug target is: **Manual line fitting with gradients, then the same loop in PyTorch.**

## Scratch → framework mapping

| From-scratch idea in this repo | Practical framework/API | What to learn from the framework |
|---|---|---|
| manual gradients | torch.autograd | automatic differentiation removes hand-derived gradients but not the optimization idea |
| lists/vectors | NumPy/PyTorch tensors | real models use tensor operations for speed and batching |
| hand SGD | torch.optim.SGD/Adam | optimizers package update rules and state |

**Framework learning rule:** do not memorize the API first. First identify which scratch concept it replaces, then learn its inputs, outputs, configuration, and failure modes.

## Real-system application

Every robot policy, filter, controller, and foundation model is an error-correction loop over states, measurements, actions, or tokens.

Ask these system questions:

1. What module produces the input to this component?
2. What module consumes its output?
3. What latency, safety, calibration, or data-format assumptions exist?
4. What metric tells me this component is good enough for the larger system?

## Failure modes and debugging

Common ways this topic can fail:

- learning rate too high diverges
- shape mismatch between vectors/matrices
- confusing loss value with gradient direction
- overfitting a tiny dataset

For each failure, write:

- **Symptom:** what would I see in logs, plots, robot behavior, or evaluation?
- **Likely cause:** what assumption broke?
- **First debug action:** what is the smallest thing to inspect?

## Mini-project and mastery checklist

**Mini-project:** Fit y = ax^2 + bx + c from scratch, then repeat with PyTorch and compare parameter values.

Mastery checklist:

- [ ] I can explain the mental model in one paragraph.
- [ ] I can write the core formula and define every symbol.
- [ ] I can run or read the scratch code and point to the core update/operation.
- [ ] I can name the production framework/API version of the same idea.
- [ ] I can describe where this topic sits in a robot/car/drone/VLA stack.
- [ ] I can name at least three failure modes and one debug action for each.

**Next study steps:** Part 1 RL overview, Part 16 attention/transformers, Part 14 control theory

## 1. Mental model

This notebook builds the minimum math intuition you need before RL, robotics, and foundation models: vectors, matrices, probability, gradients, and optimization. The goal is not to become a mathematician first; the goal is to recognize the tools when they appear in policies, filters, controllers, and neural networks.

Before code, write one sentence in your own words: *what problem does this topic solve?*

## 2. Mechanism and math

A vector is a list of numbers that represents a point, direction, state, action, embedding, or error. A matrix is a function that transforms vectors. Most learning loops optimize a loss:

\[
\theta_{t+1}=\theta_t-\alpha
\nabla_\theta L(\theta_t)
\]

where `theta` are parameters, `alpha` is the learning rate, and the gradient points uphill, so subtracting it moves downhill. Robotics uses the same pattern: reduce tracking error, reduce state-estimation error, reduce prediction error, or reduce policy loss.

## 3. From-scratch lab

We fit a line `y = w*x + b` using gradient descent. This is the smallest version of neural-network training: predict, measure error, compute gradients, update parameters.

Read every line. The code avoids clever abstractions so you can see the mechanism.

In [ ]:
# Tiny gradient descent without PyTorch.
import math

xs = [-2, -1, 0, 1, 2]
ys = [2*x + 1 for x in xs]  # true rule is y = 2x + 1

w, b = 0.0, 0.0
lr = 0.1

for step in range(30):
    # Forward pass: predictions
    preds = [w*x + b for x in xs]

    # Mean squared error
    errors = [pred - y for pred, y in zip(preds, ys)]
    loss = sum(e*e for e in errors) / len(errors)

    # Gradients of MSE wrt w and b
    dL_dw = sum(2*e*x for e, x in zip(errors, xs)) / len(xs)
    dL_db = sum(2*e for e in errors) / len(xs)

    # Parameter update
    w -= lr * dL_dw
    b -= lr * dL_db

    if step % 5 == 0:
        print(f"step={step:02d} loss={loss:.4f} w={w:.3f} b={b:.3f}")

print('final prediction for x=3:', w*3 + b)

## 3.1 Code reading guide

When you read the previous cell, do not treat it as a black box. Trace it in this order:

1. **Inputs:** what are the given numbers, observations, states, rewards, or measurements?
2. **Internal variables:** what does each variable represent physically or mathematically?
3. **Update rule:** which line is the core mechanism from the math section?
4. **Output:** what should change if the mechanism is working?
5. **Failure case:** what parameter could make the example unstable, wrong, or unsafe?

This habit is the bridge between toy examples and real robotics code: every simulator, ROS node, policy, controller, or perception model still has inputs, state, an update rule, and outputs.

## 4. Framework/practice view

In practice, PyTorch computes gradients automatically. The mechanism is identical: define parameters, compute a loss, call `backward`, and step the optimizer.

The goal is not to replace understanding with APIs. The goal is to recognize the same mechanism when a library hides the details.

In [ ]:
try:
    import torch
    x = torch.tensor(xs, dtype=torch.float32).view(-1, 1)
    y = torch.tensor(ys, dtype=torch.float32).view(-1, 1)
    model = torch.nn.Linear(1, 1)
    opt = torch.optim.SGD(model.parameters(), lr=0.1)
    for step in range(30):
        loss = ((model(x) - y) ** 2).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
    print('PyTorch prediction for x=3:', model(torch.tensor([[3.0]])).item())
except ModuleNotFoundError as e:
    print('Install torch to run the framework cell:', e)

## 4.1 Framework comparison checklist

After running or reading the framework cell, write a small mapping table for yourself:

| Question | Your answer |
|---|---|
| What object/function in the framework replaces the scratch code? |  |
| Which parameters match the math symbols? |  |
| What details does the framework hide? |  |
| What new engineering concerns appear? | installation, devices, logging, data formats, batching, safety, versioning |

This is where top-down learning becomes useful: you learn the professional API **without losing the mechanism**

## 5. Real-system connection

Everything later reuses this loop. RL changes the loss to a return/value/policy objective. Kalman filters optimize belief under uncertainty. Controllers minimize tracking error. VLA models minimize action-prediction loss over robot demonstrations.

## 6. Exercises

1. Change the true rule to `y = -3x + 0.5` and watch gradient descent adapt.
2. Make the learning rate too large. What does divergence look like?
3. Write where vectors appear in: a robot state, a drone IMU sample, a language embedding, and a policy action.

**Notebook habit:** after each exercise, add a short note explaining what changed and why it matters in a robot/car/drone/VLA stack.